# Tagalog-English Machine Translation (A3 Project)

**Student**: Htut Ko Ko  
**Course**: Natural Language Understanding  
**Task**: Tagalog (tl) <-> English (en) Translation using Transformer

## Project Overview
This notebook implements a Neural Machine Translation system using a **Transformer** architecture.
We use the **ALT (Asian Language Treebank)** dataset for Tagalog-English parallel data.
We use **SentencePiece** for subword tokenization.


In [1]:
import os
import math
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset
import sentencepiece as spm

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Using device: cuda


## 2. Data Loading (ALT Dataset)
Loading Tagalog-English pairs from ALT.

In [2]:
print("Loading ALT Dataset...")
try:
    # ALT has 'fil' for Filipino (Tagalog)
    dataset = load_dataset("alt", split="train+validation+test")
    print(f"Loaded {len(dataset)} sentences from ALT.")

    data = []
    for item in dataset:
        if 'translation' in item:
            if 'fil' in item['translation'] and 'en' in item['translation']:
                data.append({
                    'tl': item['translation']['fil'],
                    'en': item['translation']['en']
                })

    print(f"Extracted {len(data)} Tagalog-English pairs.")
except Exception as e:
    print(f"Error loading from HF: {e}")

Loading ALT Dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

alt-parallel/train-00000-of-00001.parque(…):   0%|          | 0.00/31.2M [00:00<?, ?B/s]

alt-parallel/validation-00000-of-00001.p(…):   0%|          | 0.00/1.71M [00:00<?, ?B/s]

alt-parallel/test-00000-of-00001.parquet:   0%|          | 0.00/1.79M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18088 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1019 [00:00<?, ? examples/s]

Loaded 20107 sentences from ALT.
Extracted 20107 Tagalog-English pairs.


In [3]:
df = pd.DataFrame(data)
print(df.head())

df = df.dropna(subset=['tl', 'en'])
df['tl'] = df['tl'].astype(str)
df['en'] = df['en'].astype(str)
df = df[df['tl'].str.strip() != '']
df = df[df['en'].str.strip() != '']

                                                  tl  \
0  Natalo ng Italya ang Portugal sa puntos na 31-...   
1  Si Andrea Masi ang nagsimula na makapuntos sa ...   
2  Sa kabila ng pagmamanipula sa unang kalahati n...   
3  Hindi sumuko ang Portugal at si David Penalva ...   
4  Nanguna ang Italya sa puntos na 16-5 sa kalagi...   

                                                  en  
0  Italy have defeated Portugal 31-5 in Pool C of...  
1  Andrea Masi opened the scoring in the fourth m...  
2  Despite controlling the game for much of the f...  
3  Portugal never gave up and David Penalva score...  
4  Italy led 16-5 at half time but were matched b...  


## 3. Tokenization

In [4]:
# Save texts to files
with open('train_tl.txt', 'w', encoding='utf-8') as f:
    for line in df['tl']: f.write(line + '\n')

with open('train_en_tl.txt', 'w', encoding='utf-8') as f:
    for line in df['en']: f.write(line + '\n')

# Train SentencePiece models
vocab_size = 8000
model_type = 'bpe'

print("Training Tagalog Tokenizer...")
spm.SentencePieceTrainer.train(
    input='train_tl.txt',
    model_prefix='spm_tl',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

print("Training English Tokenizer (for Tagalog pair)...")
spm.SentencePieceTrainer.train(
    input='train_en_tl.txt',
    model_prefix='spm_en_tl',
    vocab_size=vocab_size,
    model_type=model_type,
    pad_id=0, bos_id=1, eos_id=2, unk_id=3
)

sp_src = spm.SentencePieceProcessor(model_file='spm_tl.model')
sp_trg = spm.SentencePieceProcessor(model_file='spm_en_tl.model')

Training Tagalog Tokenizer...
Training English Tokenizer (for Tagalog pair)...


## 4. Dataset & Model

In [5]:
class TranslationDataset(Dataset):
    def __init__(self, df, sp_src, sp_trg):
        self.data = df
        self.sp_src = sp_src
        self.sp_trg = sp_trg

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        src_text = self.data.iloc[idx]['tl']
        trg_text = self.data.iloc[idx]['en']
        src_ids = [self.sp_src.bos_id()] + self.sp_src.encode(src_text, out_type=int) + [self.sp_src.eos_id()]
        trg_ids = [self.sp_trg.bos_id()] + self.sp_trg.encode(trg_text, out_type=int) + [self.sp_trg.eos_id()]
        return torch.tensor(src_ids), torch.tensor(trg_ids)

def collate_fn(batch):
    src_batch, trg_batch = [], []
    for src, trg in batch:
        src_batch.append(src)
        trg_batch.append(trg)
    src_pad = pad_sequence(src_batch, batch_first=True, padding_value=0)
    trg_pad = pad_sequence(trg_batch, batch_first=True, padding_value=0)
    return src_pad, trg_pad

train_dataset = TranslationDataset(df, sp_src, sp_trg)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=collate_fn)

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size,
                 d_model=256, nhead=4, num_encoder_layers=2,
                 num_decoder_layers=2, dim_feedforward=512, dropout=0.1, pad_idx=0):
        super(TransformerModel, self).__init__()
        self.d_model = d_model
        self.pad_idx = pad_idx
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer = nn.Transformer(d_model=d_model, nhead=nhead, num_encoder_layers=num_encoder_layers, num_decoder_layers=num_decoder_layers, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.fc_out = nn.Linear(d_model, trg_vocab_size)

    def forward(self, src, trg):
        src_key_padding_mask = (src == self.pad_idx)
        trg_mask = self.transformer.generate_square_subsequent_mask(trg.size(1)).to(src.device)
        src_emb = self.pos_encoder(self.src_embedding(src) * math.sqrt(self.d_model))
        trg_emb = self.pos_encoder(self.trg_embedding(trg) * math.sqrt(self.d_model))
        output = self.transformer(src=src_emb, tgt=trg_emb, tgt_mask=trg_mask, src_key_padding_mask=src_key_padding_mask)
        return self.fc_out(output)

In [7]:
model = TransformerModel(vocab_size, vocab_size).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0005)
criterion = nn.CrossEntropyLoss(ignore_index=0)

print("Starting Training...")
for epoch in range(50): # 50 Epochs for ALT
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(train_loader):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()
        output = model(src, trg[:, :-1])
        output = output.contiguous().view(-1, output.shape[-1])
        trg = trg[:, 1:].contiguous().view(-1)
        loss = criterion(output, trg)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        if i % 100 == 0: print(f"Step {i}, Loss: {loss.item():.3f}")
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.3f}")

    # Save
    torch.save(model.state_dict(), 'transformer_model_tl.pt')

Starting Training...
Step 0, Loss: 9.210
Step 100, Loss: 6.797
Step 200, Loss: 6.489
Step 300, Loss: 6.381
Epoch 1 Loss: 6.725
Step 0, Loss: 6.260
Step 100, Loss: 5.953
Step 200, Loss: 5.918
Step 300, Loss: 5.736
Epoch 2 Loss: 6.015
Step 0, Loss: 5.615
Step 100, Loss: 5.594
Step 200, Loss: 5.530
Step 300, Loss: 5.515
Epoch 3 Loss: 5.589
Step 0, Loss: 5.190
Step 100, Loss: 5.415
Step 200, Loss: 5.166
Step 300, Loss: 5.139
Epoch 4 Loss: 5.258
Step 0, Loss: 5.087
Step 100, Loss: 5.019
Step 200, Loss: 5.036
Step 300, Loss: 4.975
Epoch 5 Loss: 5.001
Step 0, Loss: 4.884
Step 100, Loss: 4.778
Step 200, Loss: 4.828
Step 300, Loss: 4.676
Epoch 6 Loss: 4.792
Step 0, Loss: 4.523
Step 100, Loss: 4.596
Step 200, Loss: 4.639
Step 300, Loss: 4.588
Epoch 7 Loss: 4.617
Step 0, Loss: 4.536
Step 100, Loss: 4.275
Step 200, Loss: 4.409
Step 300, Loss: 4.398
Epoch 8 Loss: 4.461
Step 0, Loss: 4.208
Step 100, Loss: 4.340
Step 200, Loss: 4.500
Step 300, Loss: 4.374
Epoch 9 Loss: 4.323
Step 0, Loss: 4.190
Step 

In [8]:
# Copy to app
import shutil
os.makedirs('app/models', exist_ok=True)
shutil.copy('transformer_model_tl.pt', 'app/models/transformer_model_tl.pt')
shutil.copy('spm_tl.model', 'app/models/spm_tl.model')
shutil.copy('spm_en_tl.model', 'app/models/spm_en_tl.model')
print("Models copied to app/models/")

Models copied to app/models/
